## Load the Dataset

Here we are working with the `rag-mini-bioasq` database available on hugging face.

In [ ]:
from datasets import load_dataset

qap = load_dataset("enelpol/rag-mini-bioasq", "question-answer-passages")
corpus = load_dataset("enelpol/rag-mini-bioasq", "text-corpus")

print(qap)
print(corpus)


Normalize Passages, This ensures consistency for embedding and retrieval.

In [ ]:
import re

def normalize_text(text):
    # text = text.lower() # Not lowercasing to preserve acronyms
    text = re.sub(r'\s+', ' ', text).strip()
    return text

corpus = corpus.map(lambda x: {"passage": normalize_text(x["passage"])})


Plotting the words and tokens in each "passage"

In [ ]:
import matplotlib.pyplot as plt

# count words in each passage
word_counts = [len(p.split()) for p in corpus["test"]["passage"]]

plt.figure(figsize=(10,6))
plt.hist(word_counts, bins=500, alpha=0.7, color="skyblue", edgecolor="black")
plt.xlabel("Words per passage")
plt.ylabel("Frequency")
plt.title("Distribution of passage lengths (words)")
plt.show()


# count tokens in each passage using a tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
token_counts = [len(tokenizer.encode(p, add_special_tokens=False)) for p in corpus["test"]["passage"]]

plt.figure(figsize=(10,6))
plt.hist(token_counts, bins=50, alpha=0.7, color="lightgreen", edgecolor="black")
plt.xlabel("Tokens per passage")
plt.ylabel("Frequency")
plt.title("Distribution of passage lengths (tokens)")
plt.show()


## Chunking
Split Long Passages (Chunking) using sliding windows to maintain continuity in each passage

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def chunk_passage(examples, max_tokens=256, overlap=50):
    all_chunks = {"id": [], "passage": []}
    
    for idx, passage in enumerate(examples["passage"]):
        tokens = tokenizer.encode(passage, add_special_tokens=False)
        for i in range(0, len(tokens), max_tokens - overlap):
            chunk_tokens = tokens[i:i+max_tokens]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            all_chunks["id"].append(f"{examples['id'][idx]}_{i}")
            all_chunks["passage"].append(chunk_text)
            
    return all_chunks

# Apply mapping
corpus_chunks = corpus["test"].map(
    chunk_passage, 
    batched=True, 
    remove_columns=corpus["test"].column_names
)


In [ ]:
id2passage = {row["id"]: row["passage"] for row in corpus_chunks}

In [ ]:
print(f"Original number of passages: {len(corpus['test'])}")
print(f"Number of passages after chunking: {len(corpus_chunks)}")

# Naive RAG Approach

## Baseline embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load model
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode all chunks
chunk_texts = [row["passage"] for row in corpus_chunks]
chunk_ids = [row["id"] for row in corpus_chunks]

print(f"Number of chunks to embed: {len(chunk_texts)}")
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)


## Building FAISS index

In [ ]:
d = embeddings.shape[1]  # embedding dimension
index = faiss.IndexFlatL2(d)  # L2 distance
index.add(embeddings)

print(f"FAISS index built. Total vectors indexed: {index.ntotal}")

## Naive retrieval function

In [ ]:
def retrieve_top_k(query, k=10):
    query_emb = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_emb, k)
    results = [{"id": chunk_ids[i], "passage": chunk_texts[i], "distance": float(D[0][idx])} for idx, i in enumerate(I[0])]
    return results


## Quick Stats
Overview of your chunks, passages, and questions. 

Distribution of relevant passages per question

In [ ]:
# Total original passages
print(f"Original passages: {len(corpus['test'])}")

# Total chunks
print(f"Chunks after tokenization: {len(corpus_chunks)}")

# Average chunk length in words
avg_words = np.mean([len(p.split()) for p in chunk_texts])
print(f"Average chunk length (words): {avg_words:.2f}")

# Max / Min chunk length
max_words = max(len(p.split()) for p in chunk_texts)
min_words = min(len(p.split()) for p in chunk_texts)
print(f"Max chunk words: {max_words}, Min chunk words: {min_words}")

print('-'*20)

# Number of relevant passages per question
rel_counts = [len(x) for x in qap['test']['relevant_passage_ids']]
print(f"Average relevant passages per question: {np.mean(rel_counts):.2f}")
print(f"Max relevant passages per question: {max(rel_counts)}")
print(f"Min relevant passages per question: {min(rel_counts)}")



## Explore Retrieval Results

In [ ]:
sample_q = qap['test']['question'][0]
print("Sample query:", sample_q)

results = retrieve_top_k(sample_q, k=5)
for idx, res in enumerate(results):
    print(f"\nRank {idx+1}:")
    print("Chunk ID:", res['id'])
    print("Distance:", res['distance'])
    print("Passage snippet:", res['passage'][:], "...")


## RAG Answer Generation via Ollama

In [ ]:
from ollama import chat

llm_model = "gemma3:1b"

def generate_rag_answer(query, top_k=5):
    """
    Retrieve top-k passages and generate answer using Ollama LLM.
    Returns: answer string, list of retrieved passage IDs
    """
    retrieved = retrieve_top_k(query, k=top_k)
    retrieved_text = "\n\n".join([r["passage"] for r in retrieved])
    
    prompt = f"""Answer the following question using the context passages.

Context:
{retrieved_text}

Question: {query}
Answer:"""
    
    response = chat(model=llm_model, messages=[{"role": "user", "content": prompt}])
    # Corrected access to the LLM output
    # print(response)
    # print(response.message)
    # print(response.message.content)
    
    answer = response.message.content if hasattr(response, "message") else ""
    
    retrieved_ids = [r["id"] for r in retrieved]
    return answer, retrieved_ids

## Evaluation: Recall@k and MRR

In [ ]:
def evaluate_rag(test_dataset, top_k=5):
    """
    Evaluate retrieval using Recall@k and MRR.
    test_dataset: list of dicts, each dict has 'question' and 'relevant_passage_ids'
    """
    n = len(test_dataset)
    recall_sum = 0
    mrr_sum = 0
    i = 0
    for example in test_dataset:
        i+=1
        print("Question #", i, "out of", n)
        print("Question:", example["question"])
        _, retrieved_ids = generate_rag_answer(example["question"], top_k=top_k)
        
        relevant_ids = example.get("relevant_passage_ids", [])
        # Map relevant passage IDs to their chunked IDs
        relevant_chunk_ids = []
        for rid in relevant_ids:
            rid_str = str(rid)  # convert integer ID to string
            relevant_chunk_ids.extend([cid for cid in chunk_ids if cid.startswith(rid_str)])
        
        # Compute Recall@k
        hit = any(rid in retrieved_ids for rid in relevant_chunk_ids)
        recall_sum += int(hit)
        
        # Compute MRR
        rank = next((i+1 for i, rid in enumerate(retrieved_ids) if rid in relevant_chunk_ids), 0)
        mrr_sum += 1/rank if rank > 0 else 0
    
    recall = recall_sum / n
    mrr = mrr_sum / n
    return recall, mrr


In [ ]:

test_examples = [
    {
        "question": qap['test'][i]['question'],
        "relevant_passage_ids": qap['test'][i]['relevant_passage_ids']
    }
    # for i in range(len(qap['test'])) # uncomment to run on full test set
    for i in range(20)
]

recall, mrr = evaluate_rag(test_examples, top_k=5)
print("="*30)
print(f"Naive RAG baseline (LLM): Recall@5 = {recall:.3f}, MRR = {mrr:.3f}")


# Experimenting with RAG Optimization Techniques

## Changing Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO", device='mps')

Embedding with this took ~60 mins,

In [ ]:
# Encode all chunks
chunk_texts = [row["passage"] for row in corpus_chunks]
chunk_ids = [row["id"] for row in corpus_chunks]

print(f"Number of chunks to embed: {len(chunk_texts)}")
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True, device='mps')

d = embeddings.shape[1]  # embedding dimension
index = faiss.IndexFlatL2(d)  # L2 distance
index.add(embeddings)

print(f"FAISS index built. Total vectors indexed: {index.ntotal}")
def retrieve_top_k(query, k=10):
    query_emb = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_emb, k)
    results = [{"id": chunk_ids[i], "passage": chunk_texts[i], "distance": float(D[0][idx])} for idx, i in enumerate(I[0])]
    return results

# Total original passages
print(f"Original passages: {len(corpus['test'])}")

# Total chunks
print(f"Chunks after tokenization: {len(corpus_chunks)}")

# Average chunk length in words
avg_words = np.mean([len(p.split()) for p in chunk_texts])
print(f"Average chunk length (words): {avg_words:.2f}")

# Max / Min chunk length
max_words = max(len(p.split()) for p in chunk_texts)
min_words = min(len(p.split()) for p in chunk_texts)
print(f"Max chunk words: {max_words}, Min chunk words: {min_words}")

print('-'*20)

# Number of relevant passages per question
rel_counts = [len(x) for x in qap['test']['relevant_passage_ids']]
print(f"Average relevant passages per question: {np.mean(rel_counts):.2f}")
print(f"Max relevant passages per question: {max(rel_counts)}")
print(f"Min relevant passages per question: {min(rel_counts)}")


sample_q = qap['test']['question'][0]
print("Sample query:", sample_q)

results = retrieve_top_k(sample_q, k=5)
for idx, res in enumerate(results):
    print(f"\nRank {idx+1}:")
    print("Chunk ID:", res['id'])
    print("Distance:", res['distance'])
    print("Passage snippet:", res['passage'][:], "...")

from ollama import chat

llm_model = "gemma3:1b"

def generate_rag_answer(query, top_k=5):
    """
    Retrieve top-k passages and generate answer using Ollama LLM.
    Returns: answer string, list of retrieved passage IDs
    """
    retrieved = retrieve_top_k(query, k=top_k)
    retrieved_text = "\n\n".join([r["passage"] for r in retrieved])
    
    prompt = f"""Answer the following question using the context passages.

Context:
{retrieved_text}

Question: {query}
Answer:"""
    
    response = chat(model=llm_model, messages=[{"role": "user", "content": prompt}])
    # Corrected access to the LLM output
    # print(response)
    # print(response.message)
    # print(response.message.content)
    
    answer = response.message.content if hasattr(response, "message") else ""
    
    retrieved_ids = [r["id"] for r in retrieved]
    return answer, retrieved_ids
def evaluate_rag(test_dataset, top_k=5):
    """
    Evaluate retrieval using Recall@k and MRR.
    test_dataset: list of dicts, each dict has 'question' and 'relevant_passage_ids'
    """
    n = len(test_dataset)
    recall_sum = 0
    mrr_sum = 0
    i = 0
    for example in test_dataset:
        i+=1
        print("Question #", i, "out of", n)
        print("Question:", example["question"])
        _, retrieved_ids = generate_rag_answer(example["question"], top_k=top_k)
        
        relevant_ids = example.get("relevant_passage_ids", [])
        # Map relevant passage IDs to their chunked IDs
        relevant_chunk_ids = []
        for rid in relevant_ids:
            rid_str = str(rid)  # convert integer ID to string
            relevant_chunk_ids.extend([cid for cid in chunk_ids if cid.startswith(rid_str)])
        
        # Compute Recall@k
        hit = any(rid in retrieved_ids for rid in relevant_chunk_ids)
        recall_sum += int(hit)
        
        # Compute MRR
        rank = next((i+1 for i, rid in enumerate(retrieved_ids) if rid in relevant_chunk_ids), 0)
        mrr_sum += 1/rank if rank > 0 else 0
    
    recall = recall_sum / n
    mrr = mrr_sum / n
    return recall, mrr


test_examples = [
    {
        "question": qap['test'][i]['question'],
        "relevant_passage_ids": qap['test'][i]['relevant_passage_ids']
    }
    # for i in range(len(qap['test'])) # uncomment to run on full test set
    for i in range(20)
]

recall, mrr = evaluate_rag(test_examples, top_k=5)
print("="*30)
print(f"Naive RAG baseline (LLM): Recall@5 = {recall:.3f}, MRR = {mrr:.3f}")


We improved Recall@5 from 0.800 to 0.850, and we improved MRR from 0.652 to 0.656

## Adding a Cross Encoder for Reranking

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import faiss

# # Step 1: Embedding Model (Dense Retriever)
# embed_model = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO", device='mps')

# # Step 2: Encode all chunks
# chunk_texts = [row["passage"] for row in corpus_chunks]
# chunk_ids = [row["id"] for row in corpus_chunks]

# print(f"Number of chunks to embed: {len(chunk_texts)}")
# embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

# Step 3: Build FAISS Index
d = embeddings.shape[1]  # embedding dimension
index = faiss.IndexFlatL2(d)
index.add(embeddings)
print(f"FAISS index built. Total vectors indexed: {index.ntotal}")

# Step 4: Load Cross-Encoder (Re-Ranker)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device='mps')

# Step 5: Retrieve + Re-rank
def retrieve_and_rerank(query, initial_k=50, final_k=10):
    # Dense retrieval first
    query_emb = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_emb, initial_k)
    candidates = [{"id": chunk_ids[i], "passage": chunk_texts[i], "distance": float(D[0][idx])} 
                  for idx, i in enumerate(I[0])]
    
    # Prepare pairs for re-ranking
    pairs = [(query, c["passage"]) for c in candidates]
    scores = reranker.predict(pairs)
    
    # Attach scores and sort
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    
    return reranked[:final_k]

# Step 6: LLM Answer Generation
from ollama import chat
llm_model = "gemma3:1b"

def generate_rag_answer(query, top_k=5):
    retrieved = retrieve_and_rerank(query, initial_k=50, final_k=top_k)
    retrieved_text = "\n\n".join([r["passage"] for r in retrieved])
    
    prompt = f"""Answer the following question using the context passages.

Context:
{retrieved_text}

Question: {query}
Answer:"""
    
    response = chat(model=llm_model, messages=[{"role": "user", "content": prompt}])
    answer = response.message.content if hasattr(response, "message") else ""
    
    retrieved_ids = [r["id"] for r in retrieved]
    return answer, retrieved_ids

# Step 7: Evaluation
def evaluate_rag(test_dataset, top_k=5):
    n = len(test_dataset)
    recall_sum, mrr_sum = 0, 0
    
    for idx, example in enumerate(test_dataset, 1):
        print(f"Question {idx}/{n}: {example['question']}")
        _, retrieved_ids = generate_rag_answer(example["question"], top_k=top_k)
        
        relevant_ids = example.get("relevant_passage_ids", [])
        relevant_chunk_ids = []
        for rid in relevant_ids:
            rid_str = str(rid)
            relevant_chunk_ids.extend([cid for cid in chunk_ids if cid.startswith(rid_str)])
        
        # Recall@k
        hit = any(rid in retrieved_ids for rid in relevant_chunk_ids)
        recall_sum += int(hit)
        
        # MRR
        rank = next((i+1 for i, rid in enumerate(retrieved_ids) if rid in relevant_chunk_ids), 0)
        mrr_sum += 1/rank if rank > 0 else 0
    
    recall = recall_sum / n
    mrr = mrr_sum / n
    return recall, mrr

# Step 8: Run Evaluation
test_examples = [
    {
        "question": qap['test'][i]['question'],
        "relevant_passage_ids": qap['test'][i]['relevant_passage_ids']
    }
    for i in range(20)  # sample subset
]

recall, mrr = evaluate_rag(test_examples, top_k=5)
print("="*30)
print(f"RAG with Re-Ranking: Recall@5 = {recall:.3f}, MRR = {mrr:.3f}")


Increased Recall@5 from 0.800 -> 0.850 -> 0.900, and increased MRR from 0.652 -> 0.656 -> 0.817

In [ ]:
# Step 9: Interactive Q&A
if __name__ == "__main__":
    while True:
        query = input("\nEnter your question (or type 'exit' to quit): ")
        if query.lower() in ["exit", "quit", "q"]:
            break
        answer, retrieved_ids = generate_rag_answer(query, top_k=5)
        print("\n=== Answer ===")
        print(answer)
        print("\nRetrieved IDs:", retrieved_ids)
